# O que este documento faz

Antes de comparar dois anos do Censo Escolar é preciso saber *o que dá para
comparar*. Este documento responde a três perguntas, nesta ordem, e guarda a
resposta num arquivo:

1. Que variáveis existem em **todos** os anos que temos em disco?
2. Que tipo cada uma tem — segundo os dados, não segundo o nome dela?
3. Como fica o dicionário que descreve essas variáveis, com a redação oficial
   do INEP?

O produto final é `data/processed/dicionario_colunas_comuns.csv` e um mapa
`coluna -> dtype` pronto para alimentar a leitura de um painel de vários anos.
Rode este documento **antes** do `01-panorama-escolas`; ele é o que torna a série
histórica defensável.

Nenhuma lógica mora aqui — tudo vem de `censo_escolar.esquema`.

In [ ]:
import pandas as pd

from censo_escolar import (
    anos_disponiveis,
    carregar_anos,
    colunas_comuns,
    contagem_por_ano,
    contar_escolas,
    dicionario_de_dados,
    dtypes_para_serie_historica,
    matriz_presenca,
    perfilar,
    perfilar_anos,
    resumo_presenca,
    salvar_dicionario,
    tipos_por_ano,
)
from censo_escolar import plots

plots.aplicar_estilo()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

ANOS = anos_disponiveis()
ANOS

Se a lista vier vazia (ou curta), baixe o que falta antes de continuar:
`make dados ANO=2019`, `make dados ANO=2020`, e assim por diante.

# 1. Que variáveis cada ano traz

A matriz de presença lê só os cabeçalhos dos CSVs — roda em segundos mesmo com
todos os anos, porque nenhum dado é carregado.

In [ ]:
presenca = matriz_presenca(ANOS)
presenca.shape

In [ ]:
presenca.head(10)

O número de variáveis por ano já conta a história do pacote do INEP: cresce
devagar de 2019 a 2024 e despenca em 2025, quando o arquivo único de escolas foi
partido em uma tabela por entidade (veja `docs/microdados-2025.org`).

In [ ]:
contagem_por_ano(presenca)

In [ ]:
plots.barras(
    contagem_por_ano(presenca),
    x="ano",
    y="variaveis",
    titulo="Variáveis no arquivo de escolas, por edição do Censo",
    rotulo_y="variáveis",
    arquivo="variaveis_por_ano.png",
)

# 2. Quais sobrevivem à interseção

Só as variáveis presentes em **todos** os anos comparados entram numa série
histórica sem ressalva.

In [ ]:
comuns = colunas_comuns(ANOS)
len(comuns)

Vale medir o preço de incluir 2025 no recorte:

In [ ]:
{
    "todos os anos": len(colunas_comuns(ANOS)),
    "até 2024": len(colunas_comuns([a for a in ANOS if a <= 2024])),
}

A diferença não é ruído: são as variáveis que saíram da tabela de escolas em
2025 e foram para as tabelas de matrícula, docente e turma. Quem precisa delas
tem duas saídas — restringir a série a ≤ 2024, ou juntar as tabelas de 2025 pelo
`CO_ENTIDADE`.

In [ ]:
resumo = resumo_presenca(ANOS)
resumo[resumo.anos_ausentes == "2025"].head(15)[["coluna", "anos_presentes"]]

A coluna `continua` separa dois casos que parecem o mesmo e não são: variável
que nasceu tarde (série curta, porém íntegra) e variável que sumiu num ano *do
meio* (série que parece completa e tem buraco). O segundo caso é o perigoso;
aqui ele deve vir vazio.

In [ ]:
resumo[~resumo.continua][["coluna", "anos_presentes", "anos_ausentes"]]

# 3. De que tipo é cada variável

O prefixo do nome sugere o tipo (`IN_` e `TP_` são inteiros pequenos, `QT_`
conta, `CO_` codifica) e é isso que o `carregar_escolas` usa. Mas o prefixo
mente às vezes, então aqui o tipo vem **dos valores**: o arquivo é lido inteiro
como texto e cada coluna é descrita pelo que de fato apareceu nela.

In [ ]:
perfil = perfilar(2023)          # varredura completa; ~15 s
perfil.shape

In [ ]:
perfil.head(8)[
    ["coluna", "natureza", "dtype_dados", "dtype_prefixo", "pct_nulo", "n_distintos", "exemplos"]
]

`conflito` marca só o que faz a heurística **perder** informação — o ponto em que
a leitura com dtypes declarados cairia no modo tolerante do `loading.py`:

In [ ]:
perfil[perfil.conflito][["coluna", "dtype_dados", "dtype_prefixo", "exemplos"]]

Em 2023 esta tabela sai **vazia**, e é o resultado que se espera de um ano já
conferido: a única armadilha conhecida do arquivo de escolas,
`CO_ORGAO_REGIONAL`, já está registrada em `loading._CODIGOS_TEXTO`. Num ano
recém-publicado, porém, é a primeira célula a olhar — uma linha aqui é uma
coluna que o `carregar_escolas` vai ler em modo tolerante, com aviso.

O contrário — prefixo largo demais — não é erro, é memória desperdiçada. Um
`CO_UF` lido como `Int64` gasta oito bytes para guardar 27 valores:

In [ ]:
economia = perfil[(perfil.dtype_prefixo != "") & (perfil.dtype_dados != perfil.dtype_prefixo)]
economia[["coluna", "dtype_prefixo", "dtype_dados", "minimo", "maximo"]].head(12)

# 4. Tipos que mudam de um ano para o outro

Este é o problema que só aparece quando se olha vários anos ao mesmo tempo: o
mesmo nome, tipos diferentes. Concatenar `Int64` com `string` devolve uma coluna
`object`, e nada reclama.

In [ ]:
tipos_por_ano(                            # lê os sete arquivos inteiros; ~30 s
    perfilar_anos(
        ANOS,
        ["NU_CNPJ_ESCOLA_PRIVADA", "CO_ORGAO_REGIONAL", "QT_PROF_GESTAO", "TP_DEPENDENCIA"],
    )
)

Três comportamentos diferentes numa tabela só:

- `NU_CNPJ_ESCOLA_PRIVADA` é texto em todo ano **menos um**. Onde ele vira número,
  os zeros à esquerda do CNPJ somem — e um CNPJ sem o zero da frente não casa
  mais com o CNPJ dos outros anos.
- `QT_PROF_GESTAO` é sempre inteiro; o que muda é a faixa de valores. Aqui basta
  promover ao mais largo.
- `TP_DEPENDENCIA` não muda. É o caso comum, e é o que se espera de uma variável
  boa para série histórica.

# 5. O dicionário de dados

Junta as três fontes: presença de cada variável em cada ano, perfil dos valores
em cada ano, e a descrição oficial da planilha `ANEXO I - Dicionário de Dados`
que vem dentro do próprio ZIP do INEP.

In [ ]:
dicionario = dicionario_de_dados(ANOS)   # varre os CSVs de todos os anos; ~1 min
dicionario.shape

In [ ]:
dicionario.head(12)[
    ["coluna", "descricao", "natureza", "dtype_serie", "tipo_estavel", "n_distintos", "exemplos"]
]

`dtype_serie` é a resposta prática: o dtype que comporta a variável em **todos**
os anos. Onde os anos discordam, `tipo_estavel` é falso e `tipos_por_ano` mostra
quem discordou de quem.

In [ ]:
dicionario[~dicionario.tipo_estavel][
    ["coluna", "descricao", "dtype_serie", "tipos_por_ano"]
]

Para as categóricas, a planilha do INEP traz o domínio dos códigos — é o que
transforma um `3` em "Municipal":

In [ ]:
dicionario[dicionario.categorias != ""][["coluna", "descricao", "categorias"]].head(10)

Vale conferir o que está quase sempre vazio antes de contar com a variável:

In [ ]:
dicionario.nlargest(10, "pct_nulo_max")[["coluna", "descricao", "pct_nulo_max", "n_anos"]]

In [ ]:
salvar_dicionario(dicionario)

O mesmo arquivo sai pela linha de comando, sem abrir o notebook:

```shell
make dicionario              # ou: .venv/bin/censo dicionario
.venv/bin/censo comuns 2019 2020 2021 2022 2023 2024
```

# 6. Usando o dicionário: um painel de vários anos

O fecho do documento. O dicionário vira um mapa `coluna -> dtype` que se entrega
direto ao pandas:

In [ ]:
dtypes = dtypes_para_serie_historica(dicionario)
len(dtypes), list(dtypes.items())[:5]

E o painel se monta só com variáveis que o dicionário certificou como presentes
em todos os anos:

In [ ]:
chaves = [
    "NU_ANO_CENSO",
    "CO_ENTIDADE",
    "SG_UF",
    "TP_DEPENDENCIA",
    # Sem esta, o `apenas_ativas=True` do carregar_escolas não tem em que se
    # apoiar e passa direto — o painel viria com escolas extintas dentro, sem
    # aviso nenhum.
    "TP_SITUACAO_FUNCIONAMENTO",
]
assert all(c in dtypes for c in chaves)

painel = carregar_anos(ANOS, colunas=chaves)
painel.shape

In [ ]:
por_ano = contar_escolas(painel, "NU_ANO_CENSO").sort_values("NU_ANO_CENSO")
por_ano

In [ ]:
plots.linhas(
    por_ano,
    x="NU_ANO_CENSO",
    y="escolas",
    titulo="Escolas em atividade, por edição do Censo",
    rotulo_y="escolas",
    arquivo="escolas_por_ano.png",
)

Repare que esta série usa **contagem de escolas**, não `QT_MAT_BAS`. Não é
preferência: matrícula não está entre as comuns a todos os anos, e o dicionário
é justamente o que impede a gente de descobrir isso depois do gráfico pronto.

# Próximos passos

- Juntar as tabelas de 2025 (`Tabela_Matricula_2025_V2.csv` e companhia) por
  `CO_ENTIDADE`, devolvendo as `QT_*` ao conjunto comum e alongando a série.
- Fixar o dicionário num teste de regressão: se uma edição nova do INEP mudar o
  tipo de uma variável, o build reclama antes da análise.
- Estender o perfil às outras entidades (matrícula, docente, turma) —
  `ler_dicionario_inep` já aceita `aba=` para as abas novas de 2025.